# FEATURE ABLATION

In [ ]:
import pandas as pd
import torch
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score

# Import project-specific modules (based on full_feature_integration_study.ipynb)
from data_preparation import prepare_dataset, get_loss_weights
from graph_construction import build_graph
from train_and_evaluate import train_and_evaluate
from models import GoldenTransformer
from resources import (
    DATA_PATH,
    DEVICE,
    OUTPUT_TABLES_PATH,
    OUTPUT_FIGURES_PATH
)

# 1. SETUP & DATA LOADING
print("Initializing Feature Ablation Study...")
x, y, pos_combined, pos_spatial, pos_temporal = prepare_dataset(DATA_PATH)
class_weights = get_loss_weights(y=y)

# Load results from the previous "Golden" study to identify the champion config
best_transformer_df = pd.read_csv(os.path.join(OUTPUT_TABLES_PATH, "best_transformer_df.csv"))
best_row = best_transformer_df.sort_values(by='F1', ascending=False).iloc[0]

BEST_K = int(best_row['K']) # 21
BEST_HEADS = int(best_row['Heads']) # 4
BASELINE_F1 = best_row['F1'] # 0.7247

# Define the features and their order as used in Stage 3
feature_names = ['Log-Area', 'Latitude', 'Longitude', 'Days Since Start']

# Prepare the full tensor
log_area_t = x.to(DEVICE) if x.dim() == 2 else x.unsqueeze(1).to(DEVICE)
x_full = torch.cat([
    log_area_t, 
    pos_spatial.to(DEVICE), 
    pos_temporal.to(DEVICE)
], dim=1)

# 2. FEATURE ABLATION (LEAVE-ONE-OUT)
ablation_results = []
NUM_RUNS = 5 # Use same rigor as the final showdown
seeds = [111, 333, 555, 777, 999]

print(f"\nBaseline Performance (Full Features): F1 = {BASELINE_F1:.4f}")
print("-" * 60)

for i, f_name in enumerate(feature_names):
    print(f"Ablating Feature: {f_name}...")
    
    # Create mask to exclude one feature
    mask = torch.ones(x_full.size(1), dtype=torch.bool)
    mask[i] = False
    x_ablated = x_full[:, mask]

    run_scores = []
    for run in range(NUM_RUNS):
        # Rebuild graph for each run to ensure consistency
        data_ablated = build_graph('multirelational', BEST_K, pos_spatial, pos_temporal, pos_combined, x_ablated, y).to(DEVICE)
        
        # Re-initialize model with adjusted input dimension (3 features instead of 4)
        model = GoldenTransformer(
            input_dim=data_ablated.x.size(1), 
            hidden_dim=16, 
            edge_dim=data_ablated.edge_attr.size(1), 
            num_heads=BEST_HEADS
        ).to(DEVICE)

        f1, _, _ = train_and_evaluate(model, data_ablated, DEVICE, class_weights, max_epochs=500, patience=30)
        run_scores.append(f1)

    mean_f1 = np.mean(run_scores)
    std_f1 = np.std(run_scores)
    f1_drop = BASELINE_F1 - mean_f1
    relative_importance = (f1_drop / BASELINE_F1) * 100

    print(f"  -> Mean F1: {mean_f1:.4f} +/- {std_f1:.4f} | Drop: {f1_drop:.4f} ({relative_importance:+.2f}%)")
    
    ablation_results.append({
        'Feature_Ablated': f_name,
        'Mean_F1': mean_f1,
        'Std_F1': std_f1,
        'F1_Drop': f1_drop,
        'Importance_Percent': relative_importance
    })

# 3. REPORTING & VISUALIZATION
ablation_report_df = pd.DataFrame(ablation_results).sort_values(by='Importance_Percent', ascending=False)
ablation_report_df.to_csv(os.path.join(OUTPUT_TABLES_PATH, "feature_ablation_results.csv"), index=False)

# Plotting the Importance Bar Chart
plt.figure(figsize=(10, 6), dpi=300)
sns.set_theme(style="whitegrid", context="paper", font_scale=1.4)
ax = sns.barplot(data=ablation_report_df, x='Importance_Percent', y='Feature_Ablated', palette='flare')

plt.title("Feature Importance: Percentage Drop in F1 Score (LOO Ablation)", fontweight='bold')
plt.xlabel("Predictive Contribution (% Degradation in F1 when removed)")
plt.ylabel("Event Attribute")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FIGURES_PATH, "09_feature_ablation_importance.png"))
plt.show()

print("\n--- Feature Ablation Study Complete ---")
print(ablation_report_df[['Feature_Ablated', 'F1_Drop', 'Importance_Percent']].to_string(index=False))

: 